In [ ]:
employees_df = pd.read_csv(DATA_DIR / "employees.csv")
availability_df = pd.read_csv(DATA_DIR / "employee_availability.csv")
shifts_df = pd.read_csv(DATA_DIR / "shifts.csv")

calendar_df = pd.read_csv(DATA_DIR / "calendar.csv")
daily_revenue_df = pd.read_csv(DATA_DIR / "daily_revenue.csv")

In [ ]:
employees_df["hire_date"] = pd.to_datetime(employees_df["hire_date"])
employees_df["employment_end_date"] = pd.to_datetime(
    employees_df["employment_end_date"]
)

availability_df["date"] = pd.to_datetime(availability_df["date"])
shifts_df["date"] = pd.to_datetime(shifts_df["date"])
daily_revenue_df["date"] = pd.to_datetime(daily_revenue_df["date"])

In [ ]:
# Labour cost calculation
calculated_cost = (
    shifts_df["hours_worked"] *
    shifts_df["hourly_cost"]
)

difference = (
    shifts_df["labour_cost"] -
    calculated_cost
)

difference.abs().max()

In [ ]:
# Employees working outside employment period
invalid_shifts = shifts_df.merge(
    employees_df[
        ["employee_id", "hire_date", "employment_end_date"]
    ],
    on="employee_id",
    how="left"
)

invalid_shifts[
    (invalid_shifts["date"] < invalid_shifts["hire_date"]) |
    (
        invalid_shifts["employment_end_date"].notna() &
        (invalid_shifts["date"] > invalid_shifts["employment_end_date"])
    )
]

In [ ]:
daily_staffing = (
    shifts_df
    .groupby("date")
    .agg(
        employees_scheduled=("employee_id", "nunique"),
        total_hours=("hours_worked", "sum"),
        labour_cost=("labour_cost", "sum")
    )
    .reset_index()
)

In [ ]:
workforce_df = daily_revenue_df.merge(
    daily_staffing,
    on="date",
    how="left"
)

In [ ]:
workforce_df["orders_per_employee"] = (
    workforce_df["orders"] /
    workforce_df["employees_scheduled"]
)

workforce_df["revenue_per_labour_hour"] = (
    workforce_df["revenue"] /
    workforce_df["total_hours"]
)

In [ ]:
fig = px.scatter(
    workforce_df,
    x="orders",
    y="employees_scheduled",
    trendline="ols",
    title="Demand vs Employees Scheduled",
    labels={
        "orders": "Daily Orders",
        "employees_scheduled": "Employees Scheduled"
    }
)

fig.show()

In [ ]:
fig = px.line(
    workforce_df,
    x="date",
    y="labour_cost",
    title="Daily Labour Cost",
    labels={
        "date": "Date",
        "labour_cost": "Labour Cost (€)"
    }
)

fig.show()